In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [1]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))


2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


## Hiperparâmetros

In [5]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [7]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [8]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    try:
        ano_um = int(ano)
        # df_usado = f'df_ativos_{ano_um}'
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()
        # retorno_usado = retorno_usado[lista_ativos_finais]
        print("Retornos atualizados")

        sigma_usado = retorno_usado.cov()
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    # model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        return sum(model.x[a]*model.x[b]*model.sigma[a,b] for a in model.ativos for b in model.ativos)
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)
    #=============================
    # RESTRIÇÕES
    #=============================
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]  # se y=1, então x <= 0.20
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)

    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)

    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Retornos atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO DADOS DE RETORNO DE:  2015
## UTILIZANDO SIGMAS DO ANO DE:  2015
2015 -> {'ABEV3': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBAS3': 0.0, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.0, 'BEEF3': 0.0, 'BRAP4': 0.0, 'BRKM5': 0.028875503395056783, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CVCB3': 0.0, 'CYRE3': 0.038642087746687384, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EMBJ3': 0.0, 'ENGI11': 0.20000000000000012, 'EQTL3': 0.1738107627841086, 'EZTC3': 0.0, 'FLRY3': 0.1316039

In [9]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BBAS3': 0.0,
   'BBDC3': 0.0,
   'BBDC4': 0.0,
   'BBSE3': 0.0,
   'BEEF3': 0.0,
   'BRAP4': 0.0,
   'BRKM5': 0.028875503395056783,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CVCB3': 0.0,
   'CYRE3': 0.038642087746687384,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EMBJ3': 0.0,
   'ENGI11': 0.20000000000000012,
   'EQTL3': 0.1738107627841086,
   'EZTC3': 0.0,
   'FLRY3': 0.13160392389289988,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.0,
   'ITSA4': 0.0,
   'ITUB3': 0.0,
   'ITUB4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.0,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.0,
   'MULT3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'PSSA3': 0.05741488911089173,
   'RADL3': 0.0,
   'RAPT4': 0.0,
   'RENT3': 0.0,
   'SANB11': 0.0,
   'SBSP3': 0.0,
 

In [10]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo and v <= vb_peso_maximo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [11]:
df_portfolios

,ano,ativo,peso
0,2016,BRKM5,0.0289
1,2016,CYRE3,0.0386
2,2016,EQTL3,0.1738
3,2016,FLRY3,0.1316
4,2016,PSSA3,0.0574
...,...,...,...
95,2026,POMO4,0.0401
96,2026,PRIO3,0.0862
97,2026,SLCE3,0.1721
98,2026,SUZB3,0.0723


In [12]:
df_portfolios.to_csv('carteiras_minvar.csv')
